# Task 4: Neural Visual Search


## 1. Introduction

Extract hidden-layer embeddings from the selected article-type Keras classifier,
normalize them and rank gallery images by cosine similarity. No additional neural
training occurs here. The encoder may be MLP or CNN depending on Task 1 selection.
The recipe is fixed before viewing retrieval test results. Category supervision
may emphasize shape/category over colour or style; inspect errors accordingly.
The existing partitions have prior development exposure and are not an untouched test.


## 2. Library Imports & Setup

Import the required libraries and define the local model-loading and retrieval helpers. Run Task 1 first to create the selected article-type model.

In [ ]:
from IPython.display import display
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scripts.preprocessing import task_frame, select_tensorflow_device


### 2.1. Required Libraries

These libraries support the operations below; no training algorithms are imported from project scripts.

In [ ]:
from tensorflow import keras
import numpy as np
from PIL import Image
from pathlib import Path
import json
from tensorflow import keras
import hashlib
import numpy as np
from tensorflow import keras
import numpy as np
from PIL import Image


### 2.2. Saved Model Metadata

Register saved model metadata. This identity layer is needed to load the trained classifier; it does not change its outputs.

In [ ]:
@keras.utils.register_keras_serializable(package="Fashion")
class ModelMetadata(keras.layers.Layer):
    """Store label order, preprocessing and calibration inside the .keras file."""
    def __init__(self, metadata=None, **kwargs):
        super().__init__(**kwargs)
        self.metadata = dict(metadata or {})

    def call(self, inputs):
        return inputs

    def get_config(self):
        return {**super().get_config(), "metadata": dict(self.metadata)}


### 2.3. Image batch

Normalize a query image. RGB resize and normalization must match Task 1 so query and gallery vectors are comparable.

In [ ]:
def image_batch(image, image_size, mean, std):
    resized = image.convert("RGB").resize(tuple(image_size), Image.Resampling.BILINEAR)
    array = np.asarray(resized, dtype=np.float32) / 255.0
    array = (array - np.asarray(mean, dtype=np.float32)) / np.asarray(std, dtype=np.float32)
    return array[None, ...]


### 2.4. Resolve classifier path

Check that Task 1 exported its selected model before attempting retrieval.

In [ ]:
def resolve_classifier_path(path):
    path = Path(path)
    if path.exists():
        return path
    raise FileNotFoundError(f"Missing trained classifier: {path}. Run the classification notebook first.")


### 2.5. Load checkpoint

Load the saved Keras classifier and its embedded preprocessing information. Training is not repeated here.

In [ ]:
def load_checkpoint(path):
    path = resolve_classifier_path(path)
    if path.suffix == ".keras":
        model = keras.models.load_model(path, compile=False)
        return {**dict(model.get_layer("metadata").metadata), "model": model}
    raise ValueError(f"Unsupported classifier format: {path.suffix}")


### 2.6. File digest

Hash the exact encoder file. The application can then reject an index built with different weights.

In [ ]:
def file_digest(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


### 2.7. Neural Image Encoder

Extract the last hidden layer rather than class logits. Normalize each vector to unit length so its dot product gives cosine similarity.

In [ ]:
class NeuralImageEncoder:
    def __init__(self, path):
        checkpoint = load_checkpoint(path)
        model = checkpoint["model"]
        # All project classifiers end with hidden Dense, Dropout, logits, metadata.
        self.encoder = keras.Model(model.inputs, model.layers[-3].output)
        self.metadata = checkpoint

    def encode(self, images):
        inputs = np.concatenate([image_batch(image, self.metadata["image_size"],
                    self.metadata["mean"], self.metadata["std"]) for image in images])
        values = np.asarray(self.encoder(inputs, training=False), dtype=np.float32)
        return values / np.maximum(np.linalg.norm(values, axis=1, keepdims=True), 1e-12)


### 2.8. Embed images

Encode images in batches. Keeping only one batch of decoded images bounds inference memory use.

In [ ]:
def embed_images(frame, encoder, batch_size=64):
    features = []
    for start in range(0, len(frame), batch_size):
        images = []
        for path in frame.image_path.iloc[start:start + batch_size]:
            with Image.open(path) as image:
                images.append(image.convert("RGB"))
        features.append(encoder.encode(images))
    return np.vstack(features)


### 2.9. Retrieval metrics

Measure retrieval quality. Precision counts relevant results among the top k; recall divides by all relevant gallery items; reciprocal rank rewards an early first match.

In [ ]:
def retrieval_metrics(queries, gallery, query_labels, gallery_labels, k_values=(1, 5, 10)):
    """Exact cosine ranking of unit vectors, with stable gallery-order tie breaks.

    Hit rate means at least one matching article type. Recall means the fraction
    of all relevant gallery items retrieved. They are deliberately separate.
    Query batches bound score-matrix memory; the gallery is never duplicated.
    """
    if len(queries) == 0 or len(gallery) == 0:
        raise ValueError('Queries and gallery must be nonempty')
    if any(k <= 0 or k > len(gallery) for k in k_values):
        raise ValueError('Each k must be positive and no larger than the gallery')
    query_labels = np.asarray(query_labels)
    gallery_labels = np.asarray(gallery_labels)
    if len(queries) != len(query_labels) or len(gallery) != len(gallery_labels):
        raise ValueError('Feature rows and labels must have matching lengths')
    totals = {f'{metric}@{k}': 0.0 for k in k_values
              for metric in ('precision', 'hit_rate', 'recall')}
    reciprocal_rank = 0.0
    for start in range(0, len(queries), 64):
        scores = queries[start:start + 64] @ gallery.T
        rankings = np.argsort(-scores, axis=1, kind='stable')
        for offset, ranking in enumerate(rankings):
            relevant = gallery_labels[ranking] == query_labels[start + offset]
            matches = np.flatnonzero(relevant)
            if len(matches):
                reciprocal_rank += 1.0 / (matches[0] + 1)
            for k in k_values:
                hits = int(relevant[:k].sum())
                totals[f'precision@{k}'] += hits / k
                totals[f'hit_rate@{k}'] += float(hits > 0)
                totals[f'recall@{k}'] += hits / len(matches) if len(matches) else 0.0
    result = {name: value / len(queries) for name, value in totals.items()}
    result['mean_reciprocal_rank'] = reciprocal_rank / len(queries)
    return result


In [ ]:
select_tensorflow_device()
MODEL = ROOT / 'models/article_type_model.keras'
encoder = NeuralImageEncoder(MODEL)
RESULTS, FIGURES = ROOT / 'results', ROOT / 'figures'
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)


## 3. Load Metadata & Prepare Gallery Splits
Training images alone form the evaluation gallery; queries are disjoint validation/test groups.

In [ ]:
frames = {split: task_frame('articleType', split) for split in ['train', 'validation', 'test']}
for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    assert set(frames[left].group_key).isdisjoint(frames[right].group_key)
display(pd.Series({split: len(frame) for split, frame in frames.items()}))
features = {split: embed_images(frame, encoder) for split, frame in frames.items()}


## 4. Evaluate Neural Visual Search
Report precision, hit rate, recall and reciprocal rank. Joint article/colour agreement is a metadata proxy, not a human style rating. Do not tune the encoder based on these test results.

In [ ]:
def relevance(frame, joint=False):
    return (frame.articleType + '|' + frame.baseColour).to_numpy() if joint else frame.articleType.to_numpy()
rows = []
for split in ['validation', 'test']:
    for joint in [False, True]:
        metrics = retrieval_metrics(features[split], features['train'],
                    relevance(frames[split], joint), relevance(frames['train'], joint))
        rows.append({'split': split, 'relevance': 'article_and_colour' if joint else 'article', **metrics})
comparison = pd.DataFrame(rows)
display(comparison)
comparison.to_csv(RESULTS / 'visual_search_history.csv', index=False)


## 5. Visual Inspection of Retrieved Images
The first six test queries are chosen by dataset order, without choosing successful matches.

In [ ]:
queries = frames['test'].head(6)
fig, axes = plt.subplots(len(queries), 6, figsize=(15, 3 * len(queries)), squeeze=False)
for row, (_, query) in enumerate(queries.iterrows()):
    ranks = np.argsort(-(features['test'][row] @ features['train'].T), kind='stable')[:5]
    paths = [query.image_path] + frames['train'].iloc[ranks].image_path.tolist()
    for column, path in enumerate(paths):
        with Image.open(path) as image:
            axes[row, column].imshow(image.convert('RGB'))
        axes[row, column].axis('off')
        axes[row, column].set_title('Query' if column == 0 else f'Rank {column}')
fig.tight_layout()
fig.savefig(FIGURES / 'retrieval_examples.png', dpi=160)
plt.show()


## 6. Save the Search Gallery
After evaluation, include all supplied training-release partitions in the deployment gallery. The model hash prevents mismatched embeddings after retraining Task 1. The API additionally prioritizes predicted article type; the metrics above use pure cosine ranking.

In [ ]:
gallery = pd.concat(list(frames.values()), ignore_index=True)
embeddings = np.vstack(list(features.values()))
assert not gallery.id.duplicated().any()
model_dir = ROOT / 'models'
np.save(model_dir / 'visual_search_embeddings.npy', embeddings)
gallery[['id', 'image_path', 'articleType', 'baseColour', 'gender', 'usage', 'subCategory']].to_csv(
    model_dir / 'visual_search_metadata.csv', index=False)
config = {'model_type': 'keras_embedding_cosine', 'encoder_file': MODEL.name,
          'encoder_sha256': file_digest(MODEL), 'encoder_model_type': encoder.metadata['model_type'],
          'embedding_dim': embeddings.shape[1], 'recipe': 'last_hidden_layer_l2_cosine',
          'evaluation_scope': 'internal_test_with_prior_development_exposure'}
(model_dir / 'visual_search_model.json').write_text(json.dumps(config, indent=2) + '\n')
print('Exported neural search gallery and configuration.')


## 7. Observations & Discussion
Discuss validation versus test metrics, article versus joint colour agreement, rare categories and unsuccessful examples. Compare the same metrics to a relevant published retrieval method only with its dataset and protocol differences disclosed. No measured result is claimed until this notebook runs.

## 8. Export the Classification Report
Run after Tasks 1-3 have produced their summaries.

### 8.1. Required Libraries

These libraries support the operations below; no training algorithms are imported from project scripts.

In [ ]:
import json
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from scripts.preprocessing import ROOT

matplotlib.use('Agg')

DIRECTORY = ROOT / 'results'

TASKS = {'article_type': 'Article type', 'season': 'Season', 'gender': 'Gender', 'usage': 'Occasion'}

FAMILIES = {'shallow_mlp': 'Shallow MLP (baseline)', 'deeper_mlp': 'Deeper MLP',
            'cnn': 'Best CNN'}


### 8.2. Markdown table

Format measured results as a Markdown table for the report; this function does not compute new performance scores.

In [ ]:
def markdown_table(frame):
    # Avoid adding a tabulate dependency just for report generation.
    columns = [str(frame.index.name or 'Model'), *map(str, frame.columns)]
    rows = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
    rows.extend('| ' + ' | '.join([str(index), *map(str, row)]) + ' |' for index, row in frame.iterrows())
    return '\n'.join(rows)


### 8.3. Read and verify completed classification results

The final report summarizes all four targets. Run Tasks 1?3 first; missing or non-Keras summaries stop report generation.

In [ ]:
directory = ROOT / 'results'
figures_dir = ROOT / 'figures'

directory = Path(directory)

figures_dir.mkdir(parents=True, exist_ok=True)

required = [directory / f"{task}_summary.json" for task in TASKS]

if any(not path.exists() for path in required):
    raise FileNotFoundError("Train Tasks 1-3 before generating the Keras report.")

if any(json.loads(path.read_text()).get("framework") != "tensorflow_keras" for path in required):
    raise ValueError("Report requires fresh Keras results for all targets.")

comparisons = {task: pd.read_csv(directory / f'{task}_comparison.csv').set_index('family') for task in TASKS}

summaries = {task: json.loads((directory / f'{task}_summary.json').read_text()) for task in TASKS}

content = ['# MLP and CNN comparison',
           'These are measured results on the selection half of the frozen validation groups. '
           'All candidates in a task use the same images. Shallow MLP is the baseline; '
           'the three rows are trained model families. The CNN row is the best of three CNN configurations.',
           'Selection prioritizes macro-F1, then accuracy, then fewer parameters. '
           'Calibration and review policy use separate validation groups. Historical full-validation scores '
           'and results from the rice dataset are not directly comparable.']


### 8.4. Plot the three-family comparison

Show validation accuracy beside macro-F1 so class imbalance remains visible. These values come from saved results, not the test-based model selection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for axis, metric, title in zip(axes, ['validation_accuracy', 'validation_macro_f1'], ['Validation accuracy', 'Validation macro-F1']):
    table = pd.DataFrame({TASKS[task]: frame.loc[list(FAMILIES), metric].to_numpy() for task, frame in comparisons.items()},
                         index=list(FAMILIES.values()))
    content.extend([f'## {title}', markdown_table(table.map(lambda value: f'{value:.3f}'))])
    table.T.plot.bar(ax=axis, rot=0, ylim=(0, 1), title=title, legend=False)
    axis.set_ylabel(title)
    axis.grid(axis='y', alpha=0.2)

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.04))

fig.tight_layout(rect=(0, 0.09, 1, 1))

fig.savefig(figures_dir / 'classification_comparison.png', dpi=180, bbox_inches='tight')

plt.close(fig)

content.extend(['## CNN tuning',
                'Changes below compare saved best-macro-F1 checkpoints on the same selection images. '
                'Positive accuracy changes are percentage-point gains; macro-F1 changes are absolute.'])

for task, title in TASKS.items():
    tuning = pd.read_csv(directory / f'{task}_cnn_tuning.csv').set_index('method')
    table = pd.DataFrame({
        'Accuracy': tuning.validation_accuracy.map(lambda x: f'{x:.3f}'),
        'Macro-F1': tuning.validation_macro_f1.map(lambda x: f'{x:.3f}'),
        'Accuracy change (pp)': tuning.accuracy_change_vs_ordinary.map(lambda x: f'{100*x:+.2f}'),
        'Macro-F1 change': tuning.macro_f1_change_vs_ordinary.map(lambda x: f'{x:+.3f}'),
    })
    content.extend([f'### {title}', markdown_table(table)])


### 8.5. Summarize the selected models

Report the selected method and internal-test metrics, retaining the prior-exposure limitation.

In [ ]:
selected = pd.DataFrame({TASKS[task]: {
    'Selected method': summary['selected'],
    'Internal-test accuracy': f"{summary['test_metrics']['accuracy']:.3f}",
    'Internal-test macro-F1': f"{summary['test_metrics']['macro_f1']:.3f}",
} for task, summary in summaries.items()}).T

selected.index.name = 'Task'

content.extend(['## Selected-model internal test', markdown_table(selected),
                'The internal test has prior development exposure. It did not select models or hyperparameters '
                'in this experiment. These results are not independent real-world performance estimates.',
                'The results directory contains target-prefixed comparison and CNN tuning tables, '
                'and per-candidate learning histories. '
                'The web application loads the selected checkpoints directly from models/<target>_model.keras.'])

(directory / 'classification_report.md').write_text('\n\n'.join(content) + '\n', encoding='utf-8')

print(directory / 'classification_report.md')


## 9. Generate the Assignment Submission

Load the four selected classifiers exported by Tasks 1-3 and predict the supplied
course-test images. This is classification submission generation, separate from
visual-search evaluation. It does not fit models or use course-test labels.
The CSV preserves the template columns and original ID order.


### 9.1. Load the Template and Selected Models

Load each model once. Reuse this notebook's Keras loader and image preprocessing;
no prediction algorithm is imported from another project script.


In [ ]:
from scripts.preprocessing import load_prediction_template

submission_template = load_prediction_template()
submission_columns = [column for column in submission_template.columns if column != 'image_path']
submission_model_paths = {
    'articleType': ROOT / 'models/article_type_model.keras',
    'season': ROOT / 'models/season_model.keras',
    'gender': ROOT / 'models/gender_model.keras',
    'usage': ROOT / 'models/usage_model.keras',
}
submission_checkpoints = {target: load_checkpoint(path) for target, path in submission_model_paths.items()}
if set(submission_columns) != {'id', *submission_model_paths}:
    raise ValueError('Unexpected prediction-template columns')
for target, checkpoint in submission_checkpoints.items():
    if checkpoint['target'] != target:
        raise ValueError(f'Wrong target in selected model: {target}')
display(submission_template.head())


### 9.2. Predict Course-Test Images

Apply each model's saved normalization and label order. The highest logit gives
the predicted label; softmax and positive temperature scaling preserve this
argmax, so confidence calibration does not change the submission labels.
Training-only augmentation is disabled with `training=False`.


In [ ]:
submission_rows = []
for position, row in enumerate(submission_template.itertuples(index=False), start=1):
    predictions = {}
    with Image.open(row.image_path) as image:
        for target, checkpoint in submission_checkpoints.items():
            inputs = image_batch(image, checkpoint['image_size'], checkpoint['mean'], checkpoint['std'])
            logits = np.asarray(checkpoint['model'](inputs, training=False))[0]
            if not np.isfinite(logits).all():
                raise ValueError(f'Non-finite prediction for image {row.id}, target {target}')
            predictions[target] = checkpoint['labels'][int(logits.argmax())]
    submission_rows.append({'id': row.id, **predictions})
    if position % 500 == 0:
        print(f'Predicted {position:,}/{len(submission_template):,} images')
submission = pd.DataFrame(submission_rows, columns=submission_columns)
display(submission.head())


### 9.3. Validate and Save the CSV

Check ID order, duplicate IDs, missing predictions and allowed labels before
writing `prediction/styles_prediction.csv`. Rerunning this cell replaces the
previous submission. These checks verify file structure, not model accuracy.


In [ ]:
if submission.empty:
    raise ValueError('The prediction template is empty')
if submission['id'].tolist() != submission_template['id'].tolist():
    raise ValueError('Prediction IDs or order differ from the template')
if submission['id'].duplicated().any():
    raise ValueError('Submission contains duplicate IDs')
values = submission.drop(columns='id')
if values.isna().any().any() or values.eq('').any().any():
    raise ValueError('Submission contains missing predictions')
for target, checkpoint in submission_checkpoints.items():
    if not set(submission[target]).issubset(checkpoint['labels']):
        raise ValueError(f'Unknown label in {target} predictions')
submission_path = ROOT / 'prediction/styles_prediction.csv'
submission_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission):,} predictions to {submission_path}')
